In [6]:
import pandas as pd
import numpy as np


In [7]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv')

In [8]:
df.columns

Index(['Listing Title', 'area_m2', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor', 'Building_Age_Parsed',
       'age_years', 'is_new', 'age_is_lower_bound', 'is_hiraya',
       'flag_floor_above_building', 'flag_area_small', 'flag_area_large',
       'flag_fee_tiny', 'no_gratuity', 'n_rooms', 'has_L', 'has_D', 'has_K',
       'has_S', 'is_1R', 'area_per_room', 'flag_rooms_implausible',
       'flag_layout_area_conflict', 'walk_min_nearest', 'walk_min_second',
       'n_walk_stations', 'n_lines', 'walk_le_5', 'walk_le_10', 'walk_le_15',
       'has_bus_access', 'has_car_access', 'location', 'municipa

In [9]:


import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {"madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"]}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

base_maes = fold_maes(base)

with_madori_maes = fold_maes(base + groups["madori"])

diff = with_madori_maes - base_maes

print(f"Base mean MAE: {base_maes.mean():,.2f}")
print(f"With madori mean MAE: {with_madori_maes.mean():,.2f}")
print(f"Mean difference: {diff.mean():,.2f}")
print(f"Difference std: {diff.std():,.2f}")
print(f"Folds improved: {(diff < 0).sum()} / {len(diff)}")


c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [13:50:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Base mean MAE: 11,302.69
With madori mean MAE: 11,179.41
Mean difference: -123.28
Difference std: 94.34
Folds improved: 5 / 5


In [11]:
df["age_ge_45"] = (df["age_years"] >= 45).astype(int)
splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {
    "is_new": ["is_new"],
    "madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"],
    "station": ["walk_min_second","n_lines","has_bus_access"],
    "age_ge_45": ["age_ge_45"],
    "extras": ["is_multi_floor","is_hiraya","stories_below"],
}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

results = {}
for name, cols in groups.items():
    d = fold_maes(base + cols) - base_maes
    results[name] = d
    print(f"{name:10s} diff {d.mean():8.1f} ± {d.std():6.1f}  improved {(d<0).sum()}/5")


is_new     diff     17.1 ±   33.9  improved 2/5
madori     diff   -123.3 ±   94.3  improved 5/5
station    diff    -54.3 ±  160.9  improved 3/5
age_ge_45  diff      0.0 ±    0.0  improved 0/5
extras     diff    100.6 ±  128.0  improved 2/5


In [12]:
df[["is_new", "age_ge_45"]].describe()

,is_new,age_ge_45
count,183490.000000,183490.000000
mean,0.157845,0.039986
std,0.364597,0.195926
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,1.000000,1.000000


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

if "age_ge_45" not in df.columns:
    df["age_ge_45"] = (df["age_years"] >= 45).astype(int)


def make_linear(walk_mode, age_mode, extra_cols=None):
    extra_cols = extra_cols or []

    transformers = [
        ("area", Pipeline([
            ("log", FunctionTransformer(np.log)),
            ("sc", StandardScaler())
        ]), ["area_m2"]),

        ("fl", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("sc", StandardScaler())
        ]), ["floor_min", "stories_above"]),

        ("cat", OneHotEncoder(handle_unknown="ignore"), ["municipality"]),
    ]

    if walk_mode == "raw":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("sc", StandardScaler())
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "spline":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("spline", SplineTransformer(
                    n_knots=6, degree=3, knots="quantile"
                ))
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "flags":
        transformers.extend([
            (
                "walk",
                Pipeline([
                    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())
                ]),
                ["walk_min_nearest"]
            ),
            (
                "walk_flags",
                SimpleImputer(strategy="constant", fill_value=0),
                ["walk_le_5", "walk_le_10", "walk_le_15"]
            )
        ])

    if age_mode == "raw":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]),
            ["age_years"]
        ))
    elif age_mode == "spline":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("spline", SplineTransformer(
                    n_knots=7, degree=3, knots="quantile"
                ))
            ]),
            ["age_years"]
        ))

    if extra_cols:
        transformers.append((
            "extra",
            SimpleImputer(strategy="constant", fill_value=0),
            extra_cols
        ))

    feature_cols = list(dict.fromkeys(
        col for _, _, cols in transformers for col in cols
    ))

    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("pre", ColumnTransformer(transformers)),
            ("lr", LinearRegression())
        ]),
        func=np.log1p,
        inverse_func=lambda z: np.clip(np.expm1(z), 1e4, 5e6)
    ), feature_cols


runs = {
    "A": ("raw", "raw", []),
    "B": ("spline", "raw", []),
    "C": ("flags", "raw", []),
    "D": ("raw", "spline", []),
    "E": ("raw", "raw", ["age_ge_45"]),
    "F": ("raw", "spline", ["age_ge_45"]),
}

run_maes = {}
segment_maes = {}

for name, (walk_mode, age_mode, extra_cols) in runs.items():
    fold_maes = []
    fold_segment_maes = []

    model, feature_cols = make_linear(walk_mode, age_mode, extra_cols)

    for tr, te in splits:
        model.fit(df.iloc[tr][feature_cols], df.iloc[tr]["Price_yen"])
        preds = model.predict(df.iloc[te][feature_cols])

        y_test = df.iloc[te]["Price_yen"].to_numpy()
        fold_maes.append(mean_absolute_error(y_test, preds))

        if name in {"D", "F"}:
            mask = df.iloc[te]["age_years"].to_numpy() >= 45
            fold_segment_maes.append(
                mean_absolute_error(y_test[mask], preds[mask])
            )

    run_maes[name] = np.array(fold_maes)

    if name in {"D", "F"}:
        segment_maes[name] = np.array(fold_segment_maes)


a_maes = run_maes["A"]

for name, maes in run_maes.items():
    diff = maes - a_maes
    print(
        f"{name}: MAE {maes.mean():,.2f}, "
        f"mean diff {diff.mean():,.2f}, "
        f"diff std {diff.std():,.2f}, "
        f"improved {(diff < 0).sum()}/5"
    )

for name in ["D", "F"]:
    print(
        f"{name} age>=45 MAE: "
        f"{segment_maes[name].mean():,.2f}"
    )